In [1]:
import pandas as pd
import numpy as np
from collections import Counter
          
        

In [2]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None,*, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


    def is_leaf_node(self):
        return self.value is not None
    #tells you if its a leaf node or not 

In [3]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=100, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        self.n_features = (
            X.shape[1]
            if self.n_features is None
            else min(X.shape[1], self.n_features)
        )

        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Stopping criteria
        if (
            depth >= self.max_depth
            or n_labels == 1
            or n_samples < self.min_samples_split
        ):
            return Node(value=self._most_common_label(y))

        feat_idxs = np.random.choice(
            n_feats, self.n_features, replace=False
        )

        best_feature, best_thresh = self._best_split(
            X, y, feat_idxs
        )

        # Stop if no useful split exists
        if best_feature is None:
            return Node(value=self._most_common_label(y))

        left_idxs, right_idxs = self._split(
            X[:, best_feature], best_thresh
        )

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return Node(value=self._most_common_label(y))

        left = self._grow_tree(
            X[left_idxs, :], y[left_idxs], depth + 1
        )
        right = self._grow_tree(
            X[right_idxs, :], y[right_idxs], depth + 1
        )

        return Node(best_feature, best_thresh, left, right)

    def _best_split(self, X, y, feat_idxs):
        best_gain = 0
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            values = np.unique(X_column)
            thresholds = (values[:-1] + values[1:]) / 2

            for thr in thresholds:
                gain = self._information_gain(y, X_column, thr)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr

        return split_idx, split_threshold

    def _information_gain(self, y, X_column, threshold):
        parent_entropy = self._entropy(y)

        left_idxs, right_idxs = self._split(
            X_column, threshold
        )

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)

        e_l = self._entropy(y[left_idxs])
        e_r = self._entropy(y[right_idxs])

        child_entropy = (
            (n_l / n) * e_l + (n_r / n) * e_r
        )

        return parent_entropy - child_entropy

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(
            X_column <= split_thresh
        ).flatten()

        right_idxs = np.argwhere(
            X_column > split_thresh
        ).flatten()

        return left_idxs, right_idxs

    def _entropy(self, y):
     if len(y) == 0:
        return 0

     counts = np.array(list(Counter(y).values()))
     probabilities = counts / len(y)

     return -np.sum(
        probabilities * np.log2(probabilities)
    )

    def _most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]

    def predict(self, X):
        X = np.asarray(X, dtype=float)

        return np.array([
            self._traverse_tree(x, self.root) for x in X
        ])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)

        return self._traverse_tree(x, node.right)
  


In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv("yeast_cleaned.csv")

X = data.drop(columns=["name"]).to_numpy()
y = data["name"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1234
)

clf = DecisionTree(max_depth=6,min_samples_split=30)
clf.fit(X_train, y_train)

predictions = clf.predict(X_test)

def accuracy(y_test, y_pred):
    return np.mean(y_test == y_pred)

acc = accuracy(y_test, predictions)
print("Accuracy:", acc)
#getting a max accuracy of 62.88 for max depth = 6,min_samples_split=30


Accuracy: 0.6288659793814433
